# Day4_02C_Build_First_MCP_Server

## MCP Hands-on - Build the DSU Smart Campus MCP Server v1

**Duration:** 30–40 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Running Example:** DSU Smart Campus Assistant  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand what an MCP Server does.
- Create a simple MCP Server using `FastMCP`.
- Expose campus services as MCP tools.
- Understand tool names, descriptions, parameters, and return values.
- Test campus tools directly before connecting an Agent.
- Relate MCP tools to enterprise integrations.

---

## Previous Notebook Recap

In the previous notebook, we understood MCP architecture.

```text
Host Application
      |
      v
MCP Client
      |
      v
MCP Server
      |
      |-- Tools
      |-- Resources
      |-- Prompts
```

Now we will build our first MCP Server.

Not a calculator server.

Not a weather server.

We will build a practical:

```text
DSU Smart Campus MCP Server
```


# 1. What Are We Building?

Our Smart Campus Assistant needs access to campus capabilities.

For version 1, we will expose these as MCP tools:

```text
DSU Smart Campus MCP Server
      |
      |-- faculty_directory()
      |-- library_timings()
      |-- course_information()
      |-- check_lab_availability()
      |-- academic_calendar()
```

These are simple mock tools.

In a real college system, they would connect to:

- ERP
- LMS
- Library system
- Lab booking system
- Academic calendar database


# 2. Architecture

```text
Professor
   |
   v
Smart Campus Assistant
   |
   v
MCP Client
   |
   v
DSU Smart Campus MCP Server
   |
   |-- Faculty Directory Tool
   |-- Library Timings Tool
   |-- Course Information Tool
   |-- Lab Availability Tool
   |-- Academic Calendar Tool
```

In this notebook, we are building the MCP Server side.

The Agent connection will come in a later notebook.


# 3. Important Teaching Point

An MCP Server is not the AI.

It does not reason like an LLM.

It exposes capabilities that AI applications can use.

```text
LLM / Agent = decides what is needed
MCP Server = provides the capability
```

This distinction is very important.


# 4. Install Required Package

Run this in your terminal if MCP is not installed:

```bash
pip install "mcp[cli]"
```

If you are using the common FDP `requirements.txt`, make sure it includes:

```text
mcp[cli]
```

---

## Instructor Tip

Do not spend too much time on installation during the session.  
Keep this ready before the FDP.


# 5. Import FastMCP

`FastMCP` is the simplest way to build an MCP Server in Python.


In [ ]:
from mcp.server.fastmcp import FastMCP

# 6. Create the MCP Server

This server represents our DSU Smart Campus backend.


In [ ]:
mcp = FastMCP("DSU Smart Campus MCP Server")

print("MCP Server created.")

# 7. Tool 1 - Faculty Directory

This tool returns faculty details for a given department.

In a real system, this would query a database.

For teaching, we use a simple dictionary.


In [ ]:
@mcp.tool()
def faculty_directory(department: str) -> str:
    """
    Return faculty coordinator and contact details for a department.

    Args:
        department: Department name such as CSE, AI&ML, ECE, or Mechanical.

    Returns:
        Faculty coordinator details for the requested department.
    """

    data = {
        "cse": "CSE Coordinator: Prof. Kumar | Room: 402 | Email: cse.office@example.edu | Extension: 4567",
        "ai&ml": "AI&ML Coordinator: Prof. Meena | Room: 305 | Email: aiml.office@example.edu | Extension: 4588",
        "aiml": "AI&ML Coordinator: Prof. Meena | Room: 305 | Email: aiml.office@example.edu | Extension: 4588",
        "ece": "ECE Coordinator: Prof. Divya | Room: 210 | Email: ece.office@example.edu | Extension: 4599",
        "mechanical": "Mechanical Coordinator: Prof. Ramesh | Room: 115 | Email: mech.office@example.edu | Extension: 4522",
    }

    key = department.strip().lower()

    return data.get(
        key,
        "Department not found. Available departments: CSE, AI&ML, ECE, Mechanical."
    )

## What did we just learn?

We exposed a Python function as an MCP tool.

The important parts are:

- Tool name: `faculty_directory`
- Parameter: `department`
- Docstring: explains when and how to use it
- Return value: the information sent back to the AI application

The docstring is not decoration. It helps the AI understand the tool.


# 8. Tool 2 - Library Timings

This tool returns library opening hours.


In [ ]:
@mcp.tool()
def library_timings(day: str = "weekday") -> str:
    """
    Return library timings for weekdays, Saturday, or Sunday.

    Args:
        day: weekday, saturday, or sunday.

    Returns:
        Library opening and closing timings.
    """

    timings = {
        "weekday": "Library Timings: Monday to Friday, 8:00 AM to 8:00 PM.",
        "saturday": "Library Timings: Saturday, 9:00 AM to 5:00 PM.",
        "sunday": "Library is closed on Sunday except during special exam periods.",
    }

    key = day.strip().lower()

    return timings.get(
        key,
        "Please specify weekday, saturday, or sunday."
    )

# 9. Tool 3 - Course Information

This tool returns basic course information.


In [ ]:
@mcp.tool()
def course_information(course_code: str) -> str:
    """
    Return course details such as title, credits, semester, and outcomes.

    Args:
        course_code: Course code such as AI301, ML302, or AGENT401.

    Returns:
        Course details for the requested course.
    """

    courses = {
        "AI301": {
            "title": "Artificial Intelligence",
            "credits": 4,
            "semester": "5",
            "outcome": "Students understand search, reasoning, knowledge representation, and AI applications."
        },
        "ML302": {
            "title": "Machine Learning",
            "credits": 4,
            "semester": "6",
            "outcome": "Students learn supervised learning, unsupervised learning, model evaluation, and deployment basics."
        },
        "AGENT401": {
            "title": "Agentic AI Systems",
            "credits": 3,
            "semester": "7",
            "outcome": "Students learn agents, tools, planning, multi-agent systems, workflows, and MCP integration."
        }
    }

    key = course_code.strip().upper()

    if key not in courses:
        return "Course not found. Available course codes: AI301, ML302, AGENT401."

    course = courses[key]

    return (
        f"Course Code: {key}\n"
        f"Title: {course['title']}\n"
        f"Credits: {course['credits']}\n"
        f"Semester: {course['semester']}\n"
        f"Outcome: {course['outcome']}"
    )

# 10. Tool 4 - Lab Availability

This tool checks whether a lab is available.

For teaching, we simulate the response using mock data.


In [ ]:
@mcp.tool()
def check_lab_availability(lab_name: str, slot: str) -> str:
    """
    Check lab availability for a requested time slot.

    Args:
        lab_name: Lab name such as Lab 1, Lab 2, AI Lab, or IoT Lab.
        slot: Requested slot such as morning, afternoon, or evening.

    Returns:
        Availability status and next suggested slot.
    """

    availability = {
        ("ai lab", "morning"): "AI Lab is available in the morning slot.",
        ("ai lab", "afternoon"): "AI Lab is occupied in the afternoon. Next available slot: evening.",
        ("iot lab", "morning"): "IoT Lab is occupied in the morning. Next available slot: afternoon.",
        ("iot lab", "afternoon"): "IoT Lab is available in the afternoon slot.",
        ("lab 1", "morning"): "Lab 1 is available in the morning slot.",
        ("lab 2", "evening"): "Lab 2 is available in the evening slot.",
    }

    key = (lab_name.strip().lower(), slot.strip().lower())

    return availability.get(
        key,
        "Availability not found for this lab and slot. Try AI Lab, IoT Lab, Lab 1, or Lab 2 with morning, afternoon, or evening."
    )

# 11. Tool 5 - Academic Calendar

This tool returns upcoming academic events.


In [ ]:
@mcp.tool()
def academic_calendar(month: str) -> str:
    """
    Return important academic events for a given month.

    Args:
        month: Month name such as July, August, or September.

    Returns:
        Academic events planned for the requested month.
    """

    events = {
        "july": "July Events: FDP on Agentic AI, Internal Assessment 1, Research proposal review week.",
        "august": "August Events: Mid-semester examination, Placement readiness workshop, AI project review.",
        "september": "September Events: Lab evaluation, Hackathon, Industry expert lecture series.",
    }

    key = month.strip().lower()

    return events.get(
        key,
        "No events found for this month in the demo calendar. Try July, August, or September."
    )

# 12. Test the Tools Directly

Before connecting tools to an AI Agent, always test them directly.

This is a good engineering habit.


In [ ]:
print(faculty_directory("AI&ML"))
print()
print(library_timings("weekday"))
print()
print(course_information("AGENT401"))
print()
print(check_lab_availability("AI Lab", "afternoon"))
print()
print(academic_calendar("July"))

# 13. What Did We Just Build?

We created a mock MCP Server with campus tools.

```text
DSU Smart Campus MCP Server
   |
   |-- faculty_directory
   |-- library_timings
   |-- course_information
   |-- check_lab_availability
   |-- academic_calendar
```

This is the backend capability layer.

Later, an AI Agent can discover and use these tools.


# 14. Tool Metadata Matters

The AI does not magically understand tools.

It depends on tool metadata:

```text
Tool name
Tool description
Parameters
Return type
```

Example:

```python
def faculty_directory(department: str) -> str:
    """Return faculty coordinator and contact details for a department."""
```

A clear name and docstring help the AI decide when to use the tool.

---

## Poor Tool Name

```python
def get_data(x):
```

## Better Tool Name

```python
def faculty_directory(department: str):
```

Better naming improves tool selection.


# 15. Run the MCP Server

In a normal Python file, the server would be run like this:

```python
if __name__ == "__main__":
    mcp.run()
```

For notebook-based teaching, we usually show this but do not run it inside the notebook unless we are prepared to keep the process active.

In practice, you can copy the server code into a file like:

```text
smart_campus_server.py
```

and run:

```bash
python smart_campus_server.py
```


In [ ]:
# Do not run this cell inside the notebook unless you want to start the server process.

# if __name__ == "__main__":
#     mcp.run()


# 16. Save Server Code as a Python File

For the next notebook, it is useful to have a server file.

Run the following cell to generate:

```text
smart_campus_server.py
```

in the current folder.


In [ ]:
server_code = r'''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DSU Smart Campus MCP Server")

@mcp.tool()
def faculty_directory(department: str) -> str:
    """
    Return faculty coordinator and contact details for a department.
    """
    data = {
        "cse": "CSE Coordinator: Prof. Kumar | Room: 402 | Email: cse.office@example.edu | Extension: 4567",
        "ai&ml": "AI&ML Coordinator: Prof. Meena | Room: 305 | Email: aiml.office@example.edu | Extension: 4588",
        "aiml": "AI&ML Coordinator: Prof. Meena | Room: 305 | Email: aiml.office@example.edu | Extension: 4588",
        "ece": "ECE Coordinator: Prof. Divya | Room: 210 | Email: ece.office@example.edu | Extension: 4599",
        "mechanical": "Mechanical Coordinator: Prof. Ramesh | Room: 115 | Email: mech.office@example.edu | Extension: 4522",
    }
    return data.get(department.strip().lower(), "Department not found. Available departments: CSE, AI&ML, ECE, Mechanical.")

@mcp.tool()
def library_timings(day: str = "weekday") -> str:
    """
    Return library timings for weekdays, Saturday, or Sunday.
    """
    timings = {
        "weekday": "Library Timings: Monday to Friday, 8:00 AM to 8:00 PM.",
        "saturday": "Library Timings: Saturday, 9:00 AM to 5:00 PM.",
        "sunday": "Library is closed on Sunday except during special exam periods.",
    }
    return timings.get(day.strip().lower(), "Please specify weekday, saturday, or sunday.")

@mcp.tool()
def course_information(course_code: str) -> str:
    """
    Return course details such as title, credits, semester, and outcomes.
    """
    courses = {
        "AI301": ("Artificial Intelligence", 4, "5", "Search, reasoning, knowledge representation, and AI applications."),
        "ML302": ("Machine Learning", 4, "6", "Supervised learning, unsupervised learning, model evaluation, and deployment basics."),
        "AGENT401": ("Agentic AI Systems", 3, "7", "Agents, tools, planning, multi-agent systems, workflows, and MCP integration."),
    }
    key = course_code.strip().upper()
    if key not in courses:
        return "Course not found. Available course codes: AI301, ML302, AGENT401."
    title, credits, semester, outcome = courses[key]
    return f"Course Code: {key}\nTitle: {title}\nCredits: {credits}\nSemester: {semester}\nOutcome: {outcome}"

@mcp.tool()
def check_lab_availability(lab_name: str, slot: str) -> str:
    """
    Check lab availability for a requested time slot.
    """
    availability = {
        ("ai lab", "morning"): "AI Lab is available in the morning slot.",
        ("ai lab", "afternoon"): "AI Lab is occupied in the afternoon. Next available slot: evening.",
        ("iot lab", "morning"): "IoT Lab is occupied in the morning. Next available slot: afternoon.",
        ("iot lab", "afternoon"): "IoT Lab is available in the afternoon slot.",
        ("lab 1", "morning"): "Lab 1 is available in the morning slot.",
        ("lab 2", "evening"): "Lab 2 is available in the evening slot.",
    }
    return availability.get((lab_name.strip().lower(), slot.strip().lower()), "Availability not found for this lab and slot.")

@mcp.tool()
def academic_calendar(month: str) -> str:
    """
    Return important academic events for a given month.
    """
    events = {
        "july": "July Events: FDP on Agentic AI, Internal Assessment 1, Research proposal review week.",
        "august": "August Events: Mid-semester examination, Placement readiness workshop, AI project review.",
        "september": "September Events: Lab evaluation, Hackathon, Industry expert lecture series.",
    }
    return events.get(month.strip().lower(), "No events found for this month. Try July, August, or September.")

if __name__ == "__main__":
    mcp.run()
'''

with open("smart_campus_server.py", "w", encoding="utf-8") as file:
    file.write(server_code)

print("Created smart_campus_server.py")


# 17. Enterprise Mapping

The same pattern works across industries.

## Hospital MCP Server

```text
Hospital MCP Server
   |-- doctor_schedule()
   |-- ward_availability()
   |-- appointment_status()
   |-- patient_guidelines()
```

## Banking MCP Server

```text
Bank MCP Server
   |-- customer_profile()
   |-- loan_status()
   |-- branch_locator()
   |-- create_service_ticket()
```

## DevOps MCP Server

```text
DevOps MCP Server
   |-- get_logs()
   |-- get_metrics()
   |-- deployment_status()
   |-- create_incident_ticket()
```

Only the domain changes.

The MCP pattern remains the same.


# 18. Classroom Exercise

Add one more tool to the Smart Campus MCP Server.

Choose one:

```text
exam_schedule()
attendance_summary()
placement_calendar()
research_publications()
hostel_availability()
```

Suggested structure:

```python
@mcp.tool()
def exam_schedule(branch: str, semester: str) -> str:
    """Return exam schedule for branch and semester."""
    ...
```


# 19. Challenge Exercise

Create a tool called:

```python
department_information(department: str)
```

It should return:

- HOD name
- Vision
- Faculty count
- Labs
- Research areas
- Accreditation status

This will help participants think beyond simple lookup tools.


# 20. Common Mistakes

Avoid these mistakes while building MCP tools:

1. Using vague tool names.
2. Writing unclear docstrings.
3. Returning inconsistent formats.
4. Putting too much business logic inside one tool.
5. Forgetting to test tools directly.
6. Treating tools as AI.
7. Not thinking about authorization for action tools.
8. Exposing sensitive data without access control.

---

## Instructor Tip

Repeat:

> MCP Server provides capabilities. The Agent decides when to use them.


# 21. Key Takeaways

In this notebook, we learned:

- How to create a simple MCP Server using FastMCP.
- How to expose Python functions as MCP tools.
- How to build campus-related MCP tools.
- Why tool metadata matters.
- How to test tools directly.
- How MCP servers expose capabilities to AI applications.
- How the same pattern applies to banking, healthcare, DevOps, and education.

## Final Mental Model

```text
OpenAI SDK = decides
CrewAI = coordinates
LangGraph = controls
MCP Server = provides capabilities
```

Next, we will clarify a very important MCP concept:

```text
Tools vs Resources
```
